In [1]:
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input, GRU, Conv1D, MaxPool1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score, mean_squared_error, f1_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
import pickle

In [2]:
df = pd.read_csv("NIFTY.csv")
print(df.shape)
df.head()

(4977, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,NaN,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100,NaN,NaN,NaN,NaN
1,2005-11-30,2605.10009765625,2605.110107421875,2605.10009765625,2605.110107421875,0,^CNX100,NaN,NaN,NaN
2,2005-12-01,2649.75,2654.10009765625,2595.449951171875,2614.64990234375,0,^CNX100,1.713942,NaN,NaN
3,2005-12-02,2652.300048828125,2679.89990234375,2646.89990234375,2670.35009765625,0,^CNX100,0.096237,NaN,NaN
4,2005-12-05,2619.699951171875,2663.550048828125,2614.050048828125,2663.550048828125,0,^CNX100,-1.229126,NaN,NaN


In [3]:
df = df.iloc[1:4977,:]
print(df.shape)
df.head()

(4976, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
1,2005-11-30,2605.10009765625,2605.110107421875,2605.10009765625,2605.110107421875,0,^CNX100,NaN,NaN,NaN
2,2005-12-01,2649.75,2654.10009765625,2595.449951171875,2614.64990234375,0,^CNX100,1.713942,NaN,NaN
3,2005-12-02,2652.300048828125,2679.89990234375,2646.89990234375,2670.35009765625,0,^CNX100,0.096237,NaN,NaN
4,2005-12-05,2619.699951171875,2663.550048828125,2614.050048828125,2663.550048828125,0,^CNX100,-1.229126,NaN,NaN
5,2005-12-06,2618.60009765625,2646.449951171875,2604.050048828125,2622.300048828125,0,^CNX100,-0.041984,NaN,NaN


In [4]:
df.isna().sum()

Date                0
Close               0
High                0
Low                 0
Open                0
Volume              0
Ticker              0
Daily_Return_%      1
MA_50              49
MA_200            199
dtype: int64

In [5]:
colist = ['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%']
for col in colist:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"])
for col in colist:
    print(df[col].dtype)

float64
float64
float64
float64
int64
float64


#Feature Engineering

In [6]:
for col in df.columns:
    print(col, "->",df[col].dtype,)

Date -> datetime64[us]
Close -> float64
High -> float64
Low -> float64
Open -> float64
Volume -> int64
Ticker -> str
Daily_Return_% -> float64
MA_50 -> float64
MA_200 -> float64


In [7]:
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,2005-11-30,2605.100098,2605.110107,2605.100098,2605.110107,0,^CNX100,NaN,NaN,NaN
1,2005-12-01,2649.750000,2654.100098,2595.449951,2614.649902,0,^CNX100,1.713942,NaN,NaN
2,2005-12-02,2652.300049,2679.899902,2646.899902,2670.350098,0,^CNX100,0.096237,NaN,NaN
3,2005-12-05,2619.699951,2663.550049,2614.050049,2663.550049,0,^CNX100,-1.229126,NaN,NaN
4,2005-12-06,2618.600098,2646.449951,2604.050049,2622.300049,0,^CNX100,-0.041984,NaN,NaN


In [8]:
#Basic Columns
df["NIFTY_100"] = df["Close"].pct_change()
df["Volatility_100"] = (df["NIFTY_100"].rolling(20).std())

prev_close = df["Close"].shift(-1)
df["TR"] = pd.concat(
    [
        df["High"] - df["Close"], 
        (df["High"] - prev_close).abs(), 
        (df["Low"] - prev_close).abs()
    ], axis=1
).max(axis=1)

#Rolling volume
df["Rolling Volume"] = df.groupby("Ticker")["Volume"].transform(lambda x : x.rolling(20).mean())

#Rolling return
df["Rolling_Return_5D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(5))
df["Rolling_Return_10D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(10))
df["Rolling_Return_20D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(20))

#Ranges
df["High_low_%"] = ((df["High"] - df["Low"])/df["Close"])*100
df["Open_close_%"] = ((df["Close"] - df["Open"]).abs()/df["Close"])*100

#Lagged Volatility
df["Rolling_Volatility_20D"] = df.groupby("Ticker")["Daily_Return_%"].transform(lambda x: x.rolling(20).std())
df["Lagged_Volatility_1"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(1))
df["Lagged_Volatility_5"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(5))
df["Lagged_Volatility_10"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(10))

#MA_to_Price
df["MA50_to_Price"] = (df["MA_50"]/df["Close"])
df["MA200_to_Price"] = (df["MA_200"]/df["Close"])

#Volatility
df["Daily_Return"] = df.groupby("Ticker")["Close"].pct_change()

df["Rolling_Volatility_5D"] = df.groupby("Ticker")["Daily_Return"].transform(lambda x: x.rolling(5).std())
df["Rolling_Volatility_10D"] = df.groupby("Ticker")["Daily_Return"].transform(lambda x: x.rolling(10).std())

def future_shift(col):
    return col.iloc[::-1].rolling(5).std().iloc[::-1].shift(-1)

#Final prediction that is to be made
df["future_shift_5"] = df.groupby("Ticker")["Daily_Return"].transform(future_shift)

In [9]:
print(df[["Date", "Daily_Return_%", "future_shift_5"]].tail(10))

           Date  Daily_Return_%  future_shift_5
4966 2026-02-09        0.724409        0.008416
4967 2026-02-10        0.242749        0.008399
4968 2026-02-11        0.151405        0.008698
4969 2026-02-12       -0.542450        0.010744
4970 2026-02-13       -1.344713        0.009155
4971 2026-02-16        0.854460             NaN
4972 2026-02-17        0.227054             NaN
4973 2026-02-18        0.415164             NaN
4974 2026-02-19       -1.487281             NaN
4975 2026-02-20        0.485236             NaN


In [10]:
df = df.dropna()
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200,...,Rolling_Volatility_20D,Lagged_Volatility_1,Lagged_Volatility_5,Lagged_Volatility_10,MA50_to_Price,MA200_to_Price,Daily_Return,Rolling_Volatility_5D,Rolling_Volatility_10D,future_shift_5
199,2006-09-19,3348.350098,3403.899902,3331.300049,3394.800049,0,^CNX100,-1.047637,3153.158994,3058.710750,...,1.065387,1.028316,0.980829,0.697964,0.941705,0.913498,-0.010476,0.010730,0.013598,0.009519
200,2006-09-20,3388.800049,3394.350098,3314.100098,3318.750000,0,^CNX100,1.208056,3161.117993,3062.629249,...,1.053750,1.065387,1.028683,0.623630,0.932813,0.903750,0.012081,0.008237,0.014144,0.008916
201,2006-09-21,3436.300049,3439.800049,3413.500000,3413.550049,0,^CNX100,1.401676,3168.625996,3066.562000,...,1.072433,1.053750,1.029501,0.625134,0.922104,0.892402,0.014017,0.009742,0.014483,0.007153
202,2006-09-22,3427.949951,3445.550049,3410.750000,3414.649902,0,^CNX100,-0.242997,3176.421997,3070.440249,...,1.076358,1.072433,1.029120,0.626083,0.926624,0.895707,-0.002430,0.010185,0.014550,0.006808
203,2006-09-25,3407.300049,3435.649902,3400.050049,3435.250000,0,^CNX100,-0.602398,3184.655000,3074.378250,...,1.091045,1.076358,1.028316,0.999139,0.934656,0.902292,-0.006024,0.011001,0.009316,0.006344


#Training and Testing

In [11]:
X = df.drop(["future_shift_5", "Ticker", "Date", "Rolling_Volatility_5D", "Rolling_Volatility_10D", "Rolling_Volatility_20D"], axis = 1)
Y = df["future_shift_5"]

In [12]:
mxscale = MinMaxScaler()
X_scaled = mxscale.fit_transform(X)

In [13]:
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X_scaled, Y, test_size=0.2, random_state = 42)

print(Xtrain.shape)
print(Xtest.shape)
print(Ytrain.shape)
print(Ytest.shape)

(3817, 23)
(955, 23)
(3817,)
(955,)


In [ ]:
lightgbm = LGBMRegressor(
    n_estimators=1000, max_depth=20, random_state=42, min_child_samples=10
)

lightgbm.fit(Xtrain, Ytrain)

In [ ]:
params = {
    "n_estimators": [100, 500, 1000],
    "min_child_samples": [10, 20, 25],
    "min_s": [5, 10],
    "min_samples_split": [1, 2, 5],
    "max_features": [1.0, "sqrt", "log"],
    "bootstrap": [True, False],
    "criterion": ["squared_error", "absolute_error"],
    "max_samples": [None, 0.7, 0.8]
}

Lparams = {
    "n_estimators": [500, 1000],
    "max_depth": [5, 10],
    "num_leaves": [10, 20],
    "min_child_samples": [10, 20],
    "learning_rate": [0.05, 0.1]
}

In [19]:
gmodel = GridSearchCV(lightgbm, param_grid=Lparams, n_jobs=-1, cv=3)

In [20]:
gmodel.fit(Xtrain, Ytrain)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000655 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6120
[LightGBM] [Info] Number of data points in the train set: 3817, number of used features: 24
[LightGBM] [Info] Start training from score 0.010300
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMRegressor...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.05, 0.1], 'max_depth': [5, 10], 'min_child_samples': [10, 20], 'n_estimators': [500, 1000], ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verb

In [21]:
print(gmodel.best_params_)

{'learning_rate': 0.05, 'max_depth': 10, 'min_child_samples': 10, 'n_estimators': 1000, 'num_leaves': 20}


In [53]:
Ypred = lightgbm.predict(Xtest)
print("mse",mean_squared_error(Ytest, Ypred))
print("mae",mean_absolute_error(Ytest, Ypred))
print("r2score",r2_score(Ytest, Ypred)*100)

mse 1.3610052748957187e-05
mae 0.0024831404910005046
r2score 70.29780089187925


In [18]:
Ypred = gmodel.predict(Xtest)
print("mse", mean_squared_error(Ytest, Ypred))
print("mae", mean_absolute_error(Ytest, Ypred))
print("r2score", r2_score(Ytest, Ypred)*100)

NameError: name 'gmodel' is not defined

In [19]:
X.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%', 'MA_50',
       'MA_200', 'NIFTY_100', 'Volatility_100', 'TR', 'Rolling Volume',
       'Rolling_Return_5D', 'Rolling_Return_10D', 'Rolling_Return_20D',
       'High_low_%', 'Open_close_%', 'Lagged_Volatility_1',
       'Lagged_Volatility_5', 'Lagged_Volatility_10', 'MA50_to_Price',
       'MA200_to_Price', 'Daily_Return'],
      dtype='str')

In [20]:
importance = pd.Series(
    lightgbm.feature_importances_,
    index = X.columns
).sort_values(ascending=False)

In [21]:
print(importance)

TR                      2178
Daily_Return_%          2110
Rolling_Return_10D      2045
Rolling_Return_5D       1990
High_low_%              1933
Open_close_%            1849
Rolling_Return_20D      1798
MA50_to_Price           1766
MA200_to_Price          1739
Rolling Volume          1695
Lagged_Volatility_10    1648
Volume                  1617
Lagged_Volatility_5     1437
Volatility_100          1388
Lagged_Volatility_1     1236
MA_50                    865
MA_200                   820
Close                    790
High                     398
Open                     351
Low                      347
NIFTY_100                  0
Daily_Return               0
dtype: int32


#Using LSTM

In [22]:
dfcopy = df

In [23]:
cols = ['Close', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return_%', 'MA_50', 'MA_200', 'NIFTY_100', 'Volatility_100',
       'TR', 'Rolling Volume', 'Rolling_Return_5D', 'Rolling_Return_10D',
       'Rolling_Return_20D', 'High_low_%', 'Open_close_%', 'Rolling_Volatility_20D',
       'Lagged_Volatility_1', 'Lagged_Volatility_5', 'Lagged_Volatility_10',
       'MA50_to_Price', 'MA200_to_Price']

In [24]:
dfcopy.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200,...,Rolling_Volatility_20D,Lagged_Volatility_1,Lagged_Volatility_5,Lagged_Volatility_10,MA50_to_Price,MA200_to_Price,Daily_Return,Rolling_Volatility_5D,Rolling_Volatility_10D,future_shift_5
199,2006-09-19,3348.350098,3403.899902,3331.300049,3394.800049,0,^CNX100,-1.047637,3153.158994,3058.710750,...,1.065387,1.028316,0.980829,0.697964,0.941705,0.913498,-0.010476,0.010730,0.013598,0.009519
200,2006-09-20,3388.800049,3394.350098,3314.100098,3318.750000,0,^CNX100,1.208056,3161.117993,3062.629249,...,1.053750,1.065387,1.028683,0.623630,0.932813,0.903750,0.012081,0.008237,0.014144,0.008916
201,2006-09-21,3436.300049,3439.800049,3413.500000,3413.550049,0,^CNX100,1.401676,3168.625996,3066.562000,...,1.072433,1.053750,1.029501,0.625134,0.922104,0.892402,0.014017,0.009742,0.014483,0.007153
202,2006-09-22,3427.949951,3445.550049,3410.750000,3414.649902,0,^CNX100,-0.242997,3176.421997,3070.440249,...,1.076358,1.072433,1.029120,0.626083,0.926624,0.895707,-0.002430,0.010185,0.014550,0.006808
203,2006-09-25,3407.300049,3435.649902,3400.050049,3435.250000,0,^CNX100,-0.602398,3184.655000,3074.378250,...,1.091045,1.076358,1.028316,0.999139,0.934656,0.902292,-0.006024,0.011001,0.009316,0.006344


In [25]:
for ticker, data in dfcopy.groupby("Ticker"):
    Xsc = []
    Ysc = []

    X = data[cols].values
    Y = data["future_shift_5"]
    msc = MinMaxScaler()
    Xmsc = msc.fit_transform(X)

    window = 60
    for i in range(window, len(X)):
        Xsc.append(Xmsc[i-window:i])
        Ysc.append(Y.iloc[i])

    Xsc = np.array(Xsc)
    Ysc = np.array(Ysc)

In [26]:
Xsc.dtype

dtype('float64')

In [27]:
split = int(len(Xsc)*0.8)

XLTrain = Xsc[:split]
XLTest = Xsc[split:]

YLTrain = Ysc[:split]
YLTest = Ysc[split:]

In [28]:
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(
    YLTrain.reshape(-1,1)
)

yt_scaled = y_scaler.transform(
    YLTest.reshape(-1,1)
)

In [29]:
XLTrain.dtype
YLTrain.dtype

dtype('float64')

In [30]:
print("XLTrain", XLTrain.shape)
print("XLTest", XLTest.shape)
print("YLTrain", YLTrain.shape)
print("YLTest", YLTest.shape)

XLTrain (3769, 60, 23)
XLTest (943, 60, 23)
YLTrain (3769,)
YLTest (943,)


In [31]:
modelL = Sequential(
    [Input(shape=(60, XLTrain.shape[2])),
    GRU(128, return_sequences=True), 
    Dropout(0.2),
    GRU(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)]
)

modelL.compile(
    optimizer=Adam(learning_rate=0.005),
    loss='mse'
)

call_back = [EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)
    # ReduceLROnPlateau(
    #     moniter='val_loss',
    #     patience=5,
    #     factor=0.5,
    #     min_LR=1e-6
    # )
]
modelL.fit(
    XLTrain, 
    y_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=call_back
)

Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 66ms/step - loss: 0.0186 - val_loss: 0.0045
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 0.0069 - val_loss: 0.0042
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - loss: 0.0064 - val_loss: 0.0045
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 0.0063 - val_loss: 0.0046
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 0.0063 - val_loss: 0.0043
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - loss: 0.0060 - val_loss: 0.0039
Epoch 7/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - loss: 0.0059 - val_loss: 0.0046
Epoch 8/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - loss: 0.0059 - val_loss: 0.0043
Epoch 9/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 0.0058 - val_loss: 0.0048
Epoch 10/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 0.0057 - val_loss: 0.0041
Epoch 11/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 0.0058 - val_loss: 0.0040
Epoch 12/50
106/106 ━━━━━━━━━━━━━━━━━━━━

In [32]:
modelC = Sequential([
    Input(shape=(60, XLTrain.shape[2])),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPool1D(pool_size=2),
    Dropout(0.2),

    Conv1D(filters=128, kernel_size=5, activation='relu'),
    MaxPool1D(pool_size=2),
    Dropout(0.2),

    Flatten(),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(1)]
)

modelC.compile(
    optimizer=Adam(learning_rate=0.005),
    loss="mse",
    metrics=["mse"]
)

modelC.fit(
    XLTrain,
    YLTrain,
    epochs=50,
    batch_size=32,
    validation_split = 0.1,
    callbacks = call_back
)

Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0038 - mse: 0.0038 - val_loss: 3.0777e-05 - val_mse: 3.0777e-05
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 6.3010e-05 - mse: 6.3010e-05 - val_loss: 2.9238e-05 - val_mse: 2.9238e-05
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 5.7209e-05 - mse: 5.7209e-05 - val_loss: 3.0389e-05 - val_mse: 3.0389e-05
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 6.0910e-05 - mse: 6.0910e-05 - val_loss: 2.8181e-05 - val_mse: 2.8181e-05
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.8437e-05 - mse: 5.8437e-05 - val_loss: 2.8611e-05 - val_mse: 2.8611e-05
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.7493e-05 - mse: 5.7493e-05 - val_loss: 2.8244e-05 - val_mse: 2.8244e-05
Epoch 7/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.3756e-05 - mse: 5.3756e-05 - val_loss: 2.9388e-05 - val_mse: 2.9388e-05
Epoch 8/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.21

In [33]:
Ypred = modelL.predict(XLTest)
pred = y_scaler.inverse_transform(Ypred)
print("r2score",r2_score(YLTest, pred))
print("mae",mean_absolute_error(YLTest, pred))

30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
r2score -2.148666225218107
mae 0.006544558637628359


In [34]:
Ypred = modelC.predict(XLTest)
# pred = y_scaler.inverse_transform(Ypred)
print("r2score",r2_score(YLTest, Ypred))
print("mae",mean_absolute_error(YLTest, Ypred))

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
r2score -2.0778620357610214
mae 0.005758641748989651


In [35]:
print("Actual:", YLTest[:10])
print("Predicted:", Ypred[:10].flatten())

Actual: [0.01410684 0.01382924 0.01452269 0.01494132 0.01443347 0.01397927
 0.01216147 0.01322085 0.01336921 0.01020335]
Predicted: [0.00902836 0.00872875 0.00896194 0.00920199 0.0103381  0.01138048
 0.01209394 0.01196217 0.01168162 0.01097497]


In [36]:
print(XLTrain.shape, XLTest.shape)
print(YLTrain.shape, YLTest.shape)
print("Actual:", YLTest[:10])
print("Predicted:", Ypred[:10].flatten())

(3769, 60, 23) (943, 60, 23)
(3769,) (943,)
Actual: [0.01410684 0.01382924 0.01452269 0.01494132 0.01443347 0.01397927
 0.01216147 0.01322085 0.01336921 0.01020335]
Predicted: [0.00902836 0.00872875 0.00896194 0.00920199 0.0103381  0.01138048
 0.01209394 0.01196217 0.01168162 0.01097497]


In [37]:
print("Actual mean:", YLTest.mean())
print("Actual std:", YLTest.std())

print("Pred mean:", pred.mean())
print("Pred std:", pred.std())

print("Actual min/max:", YLTest.min(), YLTest.max())
print("Pred min/max:", pred.min(), pred.max())

Actual mean: 0.007059997129365867
Actual std: 0.004262675014094219
Pred mean: 0.012804042
Pred std: 0.0028643003
Actual min/max: 0.0009394142581399712 0.04211738838320969
Pred min/max: 0.008954329 0.037147686


In [38]:
baseline = np.full_like(YLTest, YLTrain.mean())

print("Baseline MAE:",
      mean_absolute_error(YLTest, baseline))

print("Baseline R2:",
      r2_score(YLTest, baseline))

Baseline MAE: 0.004993804711517428
Baseline R2: -0.8513850182717915
